# Module 2 — Working with Data
## Module Demo: End-to-End Data Pipeline

---

**The Story:**

You've been hired as a data analyst at **ShopNaija** — a growing Nigerian e-commerce startup. On your first day, the Head of Operations drops a raw CSV on your desk:

> *"Here are 500 customer orders from the last year. It's a bit messy. I need a performance report by end of day — revenue by state, our best categories, delivery rates, and anything unusual you find. Oh, and we're sharing a summary with our logistics partner tomorrow, so make sure customer details are protected."*

This notebook walks the complete pipeline from raw file to business report — using every skill from Module 2.

| Stage | Topic |
|---|---|
| Quick array stats on raw numbers | NumPy (Topic 1) |
| Load and inspect the full table | Pandas (Topic 2) |
| Fix all data quality issues | Cleaning (Topic 3) |
| Answer the business questions | Transforming (Topic 4) |
| Understand distributions and outliers | Statistics (Topic 5) |
| Protect customer PII before sharing | Privacy (Topic 6) |

---

## Stage 1 — Quick Array Stats with NumPy
*(Topic 1)*

Before loading the full table, use NumPy to get a fast read on the revenue column.

In [ ]:
import numpy as np
import csv

# Load just the total_amount column using plain CSV reader
raw_amounts = []
with open("../data/orders.csv", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        if row["total_amount"] != "":
            raw_amounts.append(float(row["total_amount"]))

amounts = np.array(raw_amounts)
print(f"Orders with amount data: {len(amounts)}")
print()
print("===== Quick Revenue Snapshot (NumPy) =====")
print(f"Total revenue:  NGN {np.sum(amounts):>12,.0f}")
print(f"Mean order:     NGN {np.mean(amounts):>12,.0f}")
print(f"Median order:   NGN {np.median(amounts):>12,.0f}")
print(f"Std deviation:  NGN {np.std(amounts):>12,.0f}")
print(f"Min order:      NGN {np.min(amounts):>12,.0f}")
print(f"Max order:      NGN {np.max(amounts):>12,.0f}")
print()
print(f"Mean vs Median gap: NGN {np.mean(amounts) - np.median(amounts):,.0f}")
print("→ Gap signals right-skewed data — outliers present")
print()
# High-value orders using Boolean indexing
high_value = amounts[amounts > 50000]
print(f"Orders above NGN 50,000: {len(high_value)} ({len(high_value)/len(amounts)*100:.1f}%)")

---
## Stage 2 — Load and Inspect the Full Table
*(Topic 2)*

Now load all 14 columns into Pandas and understand the dataset's structure.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/orders.csv", encoding="utf-8-sig")

print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head(5)

In [ ]:
# The four first-look commands — always run these on a new dataset
print("=== info() ===")
df.info()
print()
print("=== describe() ===")
df.describe().round(0)

In [ ]:
# Catalogue the problems we found
print("===== DATA QUALITY ISSUES FOUND =====")
print(f"1. Missing phone:        {df['phone'].isnull().sum()} rows")
print(f"2. Missing total_amount: {df['total_amount'].isnull().sum()} rows")
print(f"3. Duplicate rows:       {df.duplicated().sum()} rows")
print(f"4. unit_price dtype:     {df['unit_price'].dtype} (should be float)")
print(f"5. State casing:         {df['state'].nunique()} unique values (should be 10)")
print(f"6. order_date dtype:     {df['order_date'].dtype} (should be datetime)")

---
## Stage 3 — Clean the Dataset
*(Topic 3)*

Fix every problem found above — in the correct order.

In [ ]:
# Always work on a copy
df_clean = df.copy()

# 1. Missing phone → fill with placeholder
df_clean["phone"] = df_clean["phone"].fillna("Not provided")

# 2. Missing total_amount → drop rows (key metric, cannot impute)
df_clean = df_clean.dropna(subset=["total_amount"])

# 3. Remove duplicates
df_clean = df_clean.drop_duplicates()

# 4. Fix unit_price: strip ₦ → convert to float
df_clean["unit_price"] = pd.to_numeric(
    df_clean["unit_price"].str.replace("₦", "", regex=False),
    errors="coerce"
)

# 5. Fix state casing
df_clean["state"] = df_clean["state"].str.strip().str.title()

# 6. Fix order_date → datetime + extract month/quarter
df_clean["order_date"]    = pd.to_datetime(df_clean["order_date"])
df_clean["order_month"]   = df_clean["order_date"].dt.month
df_clean["order_quarter"] = df_clean["order_date"].dt.quarter

print("===== CLEANING COMPLETE =====")
print(f"Rows:       {df.shape[0]} → {df_clean.shape[0]} ({df.shape[0]-df_clean.shape[0]} removed)")
print(f"Missing:    {df_clean.isnull().sum().sum()} remaining")
print(f"Duplicates: {df_clean.duplicated().sum()} remaining")
print(f"States:     {df_clean['state'].nunique()} unique values")
print(f"unit_price: {df_clean['unit_price'].dtype}")
print(f"order_date: {df_clean['order_date'].dtype}")

---
## Stage 4 — Transform: Answer the Business Questions
*(Topic 4)*

The Head of Operations asked for: revenue by state, best categories, delivery rates, anything unusual.

In [ ]:
# Add order tier — useful segmentation column
def assign_tier(amount):
    if amount > 30000:  return "Premium"
    elif amount > 10000: return "Mid-range"
    else:                return "Standard"

df_clean["order_tier"] = df_clean["total_amount"].apply(assign_tier)
df_clean["amount_with_vat"] = (df_clean["total_amount"] * 1.075).round(2)

In [ ]:
# Q1: Revenue by state
state_perf = (
    df_clean.groupby("state")["total_amount"]
    .agg(orders="count", revenue="sum", avg_order="mean")
    .round(0)
    .sort_values("revenue", ascending=False)
)
print("===== Revenue by State =====")
state_perf

In [ ]:
# Q2: Best performing categories
category_perf = (
    df_clean.groupby("product_category")["total_amount"]
    .agg(orders="count", revenue="sum", avg_order="mean", max_order="max")
    .round(0)
    .sort_values("revenue", ascending=False)
)
print("===== Category Performance =====")
category_perf

In [ ]:
# Q3: Delivery rate overall and by category
delivery_rate = (df_clean["delivery_status"] == "Delivered").mean() * 100
print(f"Overall delivery rate: {delivery_rate:.1f}%")
print()

delivery_by_category = (
    df_clean.groupby("product_category")["delivery_status"]
    .apply(lambda x: (x == "Delivered").mean() * 100)
    .round(1)
    .sort_values(ascending=False)
    .rename("delivery_rate_%")
)
print("Delivery rate by category:")
print(delivery_by_category)

In [ ]:
# Q4: Top 10 individual orders
top10 = (
    df_clean.nlargest(10, "total_amount")
    [["order_id", "state", "product_category", "product_name",
      "quantity", "total_amount", "delivery_status"]]
    .reset_index(drop=True)
)
print("===== Top 10 Orders by Value =====")
top10

---
## Stage 5 — Statistical Analysis
*(Topic 5)*

Go deeper than summary numbers — understand the shape of the data, detect outliers, and measure relationships.

In [ ]:
amounts_clean = df_clean["total_amount"]

print("===== Statistical Profile: total_amount =====")
print(f"Mean:     NGN {amounts_clean.mean():>10,.0f}")
print(f"Median:   NGN {amounts_clean.median():>10,.0f}")
print(f"Std Dev:  NGN {amounts_clean.std():>10,.0f}")
print(f"Skewness: {amounts_clean.skew():.2f}  → {'right-skewed' if amounts_clean.skew() > 1 else 'approximately symmetric'}")
print()

q1  = amounts_clean.quantile(0.25)
q3  = amounts_clean.quantile(0.75)
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr

print(f"IQR:          NGN {iqr:>10,.0f}")
print(f"Upper fence:  NGN {upper_fence:>10,.0f}  (Q3 + 1.5 × IQR)")

In [ ]:
# Detect and investigate outliers
outliers = df_clean[df_clean["total_amount"] > upper_fence].copy()
print(f"Outliers detected: {len(outliers)} orders above NGN {upper_fence:,.0f}")
print()

# Investigate the most extreme
extreme = outliers.nlargest(5, "total_amount")
for _, row in extreme.iterrows():
    implied_unit = row["total_amount"] / row["quantity"]
    print(f"  {row['order_id']}: {row['product_category']}, "
          f"qty={row['quantity']}, total=NGN {row['total_amount']:,.0f}, "
          f"implied unit=NGN {implied_unit:,.0f} → "
          f"{'Plausible' if implied_unit < 150000 else 'Investigate'}")

print()
print("Decision: Retain outliers — values are consistent with bulk purchases.")
print("Using MEDIAN for central tendency in the final report.")

In [ ]:
# Correlation between key numeric columns
corr = df_clean[["quantity", "unit_price", "total_amount"]].corr().round(3)
print("===== Correlation Matrix =====")
print(corr)
print()
r = df_clean["quantity"].corr(df_clean["total_amount"])
print(f"quantity ↔ total_amount: r = {r:.3f}")
print("More items ordered → higher total (moderate positive relationship)")

In [ ]:
# Statistical comparison across categories — using median (data is skewed)
cat_stats = (
    df_clean.groupby("product_category")["total_amount"]
    .agg(count="count", median="median", std="std")
    .round(0)
    .sort_values("median", ascending=False)
)
print("===== Category Stats (Median Used — Data is Right-Skewed) =====")
cat_stats

---
## Stage 6 — Protect Customer Privacy Before Sharing
*(Topic 6)*

The logistics partner needs order data — but must not receive any customer personal information.

In [ ]:
import hashlib

SALT = "shopnaija_ndpa_2024"

def hash_pii(value, salt=SALT):
    return hashlib.sha256((str(value) + salt).encode()).hexdigest()[:16]

print("===== PII Audit =====")
pii_cols = ["customer_name", "phone", "email"]
for col in df_clean.columns:
    flag = "[DIRECT PII]" if col in pii_cols else "            "
    print(f"  {flag} {col}")

In [ ]:
# Pseudonymise customer_name → customer_id (logistics partner needs order linkage)
df_clean["customer_id"] = df_clean["customer_name"].apply(hash_pii)

# Build privacy-safe export — drop all direct PII
df_safe = df_clean.drop(columns=["customer_name", "phone", "email"]).copy()

# Verify
remaining = [c for c in pii_cols if c in df_safe.columns]
print(f"Direct PII remaining: {remaining if remaining else 'NONE — safe to share'}")
print(f"Shape: {df_safe.shape[0]} rows x {df_safe.shape[1]} columns")
df_safe[["order_id", "customer_id", "state", "product_category",
          "total_amount", "delivery_status"]].head(5)

In [ ]:
# Public summary — fully aggregated, no individuals
public_summary = (
    df_safe.groupby("state")
    .agg(
        orders=("order_id", "count"),
        revenue=("total_amount", "sum"),
        delivery_rate=("delivery_status",
                       lambda x: round((x == "Delivered").mean() * 100, 1))
    )
    .round({"revenue": 0})
    .sort_values("revenue", ascending=False)
)

# Apply n < 5 rule
public_summary = public_summary[public_summary["orders"] >= 5]

print("===== Public State Summary (n>=5 filter applied) =====")
public_summary

---
## Final Deliverable — The Performance Report

In [ ]:
total_rev   = df_clean["total_amount"].sum()
total_ord   = len(df_clean)
del_rate    = (df_clean["delivery_status"] == "Delivered").mean() * 100
top_state   = df_clean.groupby("state")["total_amount"].sum().idxmax()
top_cat     = df_clean.groupby("product_category")["total_amount"].sum().idxmax()
typical_ord = df_clean["total_amount"].median()

print("=" * 58)
print("        SHOPNAIJA — ANNUAL PERFORMANCE REPORT")
print("=" * 58)
print(f"  Total orders:        {total_ord}")
print(f"  Total revenue:       NGN {total_rev:,.0f}")
print(f"  Typical order value: NGN {typical_ord:,.0f}  (median)")
print(f"  Delivery rate:       {del_rate:.1f}%")
print(f"  Top state:           {top_state}")
print(f"  Top category:        {top_cat}")
print()
print("  Revenue by State (Top 5):")
for state, rev in df_clean.groupby("state")["total_amount"].sum().nlargest(5).items():
    bar = "█" * int(rev / total_rev * 40)
    print(f"  {state:<12} {bar} NGN {rev:,.0f}")
print()
print("  Category Breakdown:")
for cat, row in category_perf.iterrows():
    share = row["revenue"] / total_rev * 100
    print(f"  {cat:<20} {int(row['orders']):>3} orders | "
          f"NGN {row['revenue']:>10,.0f} ({share:.1f}%)")
print()
print("  Statistical notes:")
print(f"    Distribution: right-skewed (skewness = {df_clean['total_amount'].skew():.1f})")
print(f"    Outliers: {len(outliers)} orders above NGN {upper_fence:,.0f} — retained (legitimate bulk orders)")
print(f"    Median used for typical order — mean (NGN {df_clean['total_amount'].mean():,.0f}) inflated by outliers")
print()
print("  Privacy:") 
print("    Logistics partner export: customer_name/phone/email removed, customer_id hash only")
print("    Public summary: fully aggregated, n>=5 filter applied")
print("=" * 58)

In [ ]:
# Save all outputs
df_clean.to_csv("../data/orders_clean.csv", index=False, encoding="utf-8-sig")
df_safe.to_csv("../data/orders_safe_export.csv", index=False, encoding="utf-8-sig")
public_summary.to_csv("../data/summary_by_state.csv", encoding="utf-8-sig")

print("Outputs saved:")
print(f"  orders_clean.csv        — {df_clean.shape[0]} rows, full analysis dataset (secure)")
print(f"  orders_safe_export.csv  — {df_safe.shape[0]} rows, no PII (logistics partner)")
print(f"  summary_by_state.csv    — {len(public_summary)} state rows, aggregated (public)")

---
## Module 2 Complete

In one continuous workflow, we:

| Stage | What we did | Skill |
|---|---|---|
| 1 | Fast array stats on raw revenue column | NumPy |
| 2 | Loaded full table, ran 4-command inspection | Pandas |
| 3 | Fixed 6 data quality problems systematically | Cleaning |
| 4 | Answered 4 business questions with groupby | Transforming |
| 5 | Profiled distribution, detected outliers, measured correlation | Statistics |
| 6 | Pseudonymised PII, built safe exports, applied n<5 rule | Privacy |

**This is the complete data analyst workflow.** From raw, messy file to structured business report with protected privacy — in under 200 lines of Python.

**Module 3 — EDA & Visualisation:** We take this same dataset and bring it to life with charts, building the visual storytelling skills that make analysis communicable to any audience.